# FRC YOLO training (Runpod, PyTorch template)
**Option A:** open this notebook in Jupyter Lab on the pod and choose **Run > Run All Cells**. Keep the browser tab open the whole time.

**Option B (recommended, keeps running if the tab closes):** open a Jupyter terminal and run:
```bash
apt-get update -qq && apt-get install -y -qq tmux   # if tmux is missing
tmux new -s train
cd /workspace && pip install -q nbconvert ipykernel && jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=-1 /workspace/train_frc.ipynb 2>&1 | tee /workspace/train_frc.log
```
Detach with `Ctrl-b d`. Reattach with `tmux attach -t train`. Check progress with `tail -f /workspace/runs/frc/results.csv`.
(Put this notebook in /workspace first. All outputs go to the network volume.)

Budget: $9.84 of credits at about $0.59/hr (L4 / RTX PRO 4000). Training stops after `time=6` hours, which costs about $3.54. Setup, validation and zipping add a few minutes.
Classes, taken from `render/post.py`: **0 bumper_red, 1 bumper_blue, 2 fuel**.

## 1. GPU check + install ultralytics

In [ ]:
!nvidia-smi
!pip install -q ultralytics
import ultralytics; ultralytics.checks()

## 2. Extract dataset (skipped if already extracted)

In [ ]:
import os, subprocess, glob
TAR='/workspace/frc_dataset.tar'; RAW='/workspace/dataset'
def find_root(base):
    for r,d,f in os.walk(base):
        if 'labels_yolo' in d and 'images' in d: return r
    return None
if find_root(RAW) and len(glob.glob(os.path.join(find_root(RAW),'images','*.jpg')))>0:
    print('Already extracted, skipping')
else:
    os.makedirs(RAW, exist_ok=True)
    subprocess.run(['tar','--no-same-owner','-xf',TAR,'-C',RAW], check=True)
SRC=find_root(RAW); assert SRC, 'images/ + labels_yolo/ not found'
print('Source root:', SRC, '| jpgs:', len(glob.glob(SRC+'/images/*.jpg')))
!df -h /workspace

## 3. Seeded 90/10 split (symlinks, no copying)
Images with **no** label file are skipped and listed, since the frame may not have been post-processed. Images with an **empty** label file stay in as background negatives.

In [ ]:
import random, collections, shutil
NAMES=['bumper_red','bumper_blue','fuel']
SPLIT='/workspace/yolo'
imgs=sorted(glob.glob(os.path.join(SRC,'images','*.jpg')))
have=[p for p in imgs if os.path.exists(os.path.join(SRC,'labels_yolo',os.path.basename(p)[:-4]+'.txt'))]
missing=len(imgs)-len(have)
print(f'images {len(imgs)}, with labels {len(have)}, missing label (skipped) {missing}')
random.Random(42).shuffle(have)
nval=max(1,round(0.1*len(have))); splits={'val':have[:nval],'train':have[nval:]}
if os.path.exists(SPLIT): shutil.rmtree(SPLIT)
for s,lst in splits.items():
    os.makedirs(f'{SPLIT}/images/{s}'); os.makedirs(f'{SPLIT}/labels/{s}')
    cnt=collections.Counter(); empty=0
    for p in lst:
        b=os.path.basename(p)[:-4]; lp=os.path.join(SRC,'labels_yolo',b+'.txt')
        os.symlink(os.path.abspath(p), f'{SPLIT}/images/{s}/{b}.jpg')
        os.symlink(os.path.abspath(lp), f'{SPLIT}/labels/{s}/{b}.txt')
        rows=[l.split() for l in open(lp) if l.strip()]
        if not rows: empty+=1
        for r in rows: cnt[int(r[0])]+=1
    print(f'{s}: {len(lst)} images ({empty} background) | ' + ', '.join(f'{NAMES[k]}={cnt[k]}' for k in range(3)))
    assert set(cnt)<=set(range(3)), f'unexpected class ids {set(cnt)}'

## 4. data.yaml (class names from post.py)

In [ ]:
import yaml
DATA='/workspace/yolo/data.yaml'
yaml.safe_dump({'path':SPLIT,'train':'images/train','val':'images/val','names':{0:'bumper_red',1:'bumper_blue',2:'fuel'}}, open(DATA,'w'), sort_keys=False)
print(open(DATA).read())

## 5. Train (yolo11s, imgsz 1280, 6-hour cap)
Everything is saved under `/workspace/runs/frc` on the network volume. `last.pt` is written every epoch and `save_period=10` keeps extra snapshots.

In [ ]:
from ultralytics import YOLO
import os
W=min(8, os.cpu_count() or 4)
model=YOLO('yolo11s.pt')
model.train(data=DATA, epochs=100, time=6, patience=20, imgsz=1280, batch=-1, workers=W, cache=False,
            project='/workspace/runs', name='frc', exist_ok=True, plots=True, save=True, save_period=10, seed=42)

### Resume (only if training was interrupted; run this instead of the cell above)

In [ ]:
# from ultralytics import YOLO
# YOLO('/workspace/runs/frc/weights/last.pt').train(resume=True)

## 6. Validate, save metrics, predict, and show plots

In [ ]:
import json, csv
from ultralytics import YOLO
RUN='/workspace/runs/frc'
best=YOLO(f'{RUN}/weights/best.pt')
m=best.val(data=DATA, imgsz=1280, batch=8, plots=True, project='/workspace/runs', name='frc_val', exist_ok=True)
b=m.box
out={'overall':{'mAP50':float(b.map50),'mAP50-95':float(b.map),'precision':float(b.mp),'recall':float(b.mr)},'per_class':{}}
for i,c in enumerate(m.ap_class_index):
    p,r,ap50,ap=b.class_result(i)
    out['per_class'][m.names[int(c)]]={'mAP50':float(ap50),'mAP50-95':float(ap),'precision':float(p),'recall':float(r)}
json.dump(out, open('/workspace/runs/metrics.json','w'), indent=2)
with open('/workspace/runs/metrics.csv','w',newline='') as f:
    w=csv.writer(f); w.writerow(['class','mAP50','mAP50-95','precision','recall'])
    for k,v in [('all',out['overall'])]+list(out['per_class'].items()): w.writerow([k,v['mAP50'],v['mAP50-95'],v['precision'],v['recall']])
print(json.dumps(out, indent=2))

In [ ]:
import glob
vimgs=sorted(glob.glob('/workspace/yolo/images/val/*.jpg'))[:20]
best.predict(vimgs, imgsz=1280, conf=0.25, save=True, project='/workspace/runs', name='frc_pred', exist_ok=True)
print('saved to /workspace/runs/frc_pred')

In [ ]:
from IPython.display import Image, display
pats=['results.png','confusion_matrix*.png','*PR_curve.png','*F1_curve.png','*P_curve.png','*R_curve.png','labels*.jpg','train_batch*.jpg','val_batch*.jpg']
for d in [RUN,'/workspace/runs/frc_val']:
    for pat in pats:
        for f in sorted(glob.glob(f'{d}/{pat}')):
            print(f); display(Image(filename=f, width=900))
for f in sorted(glob.glob('/workspace/runs/frc_pred/*.jpg'))[:6]:
    print(f); display(Image(filename=f, width=900))

## 7. Zip everything to /workspace/frc_model.zip

In [ ]:
import zipfile, os
Z='/workspace/frc_model.zip'
with zipfile.ZipFile(Z,'w',zipfile.ZIP_DEFLATED) as z:
    for d in ['/workspace/runs/frc','/workspace/runs/frc_val','/workspace/runs/frc_pred']:
        for r,_,fs in os.walk(d):
            for f in fs: z.write(os.path.join(r,f), os.path.relpath(os.path.join(r,f),'/workspace/runs'))
    for f in ['/workspace/runs/metrics.json','/workspace/runs/metrics.csv','/workspace/yolo/data.yaml']:
        if os.path.exists(f): z.write(f, os.path.basename(f))
print(Z, round(os.path.getsize(Z)/1e6,1),'MB')
print(len(zipfile.ZipFile(Z).namelist()),'files')

## Done: download and stop the pod
1. In the Jupyter file browser, right-click **/workspace/frc_model.zip** and choose **Download**. It contains best.pt, last.pt, results.csv/png, all curves and confusion matrices, batch images, args.yaml, val outputs, predictions and metrics.
2. **Stop (or terminate) the pod in the Runpod console** so it stops using credits. The network volume keeps everything, but it still has a small storage charge.